# 호출어 **v6** 학습 — 새 호출어 '하이 티드'

**v5 와 무엇이 다른가: 호출어가 바뀌었다.** `재하봇` -> `하이 티드`(hi teed).
그래서 긍정·부정을 전부 새로 만들었다. 데이터 설계도 세 군데 바뀌었다.

## 왜 호출어를 바꿨나 (2026-08-25)

`재하봇`은 whisper 에 없는 고유명사(OOV)라 **힌트 없이는 41% 밖에 안 읽혔다.**
그래서 `initial_prompt` 를 줬는데, 그게 유튜브 소리에도 '재하봇'을 만들어 냈다(환각).
환각을 막으려 2패스를 돌렸고 검증 1회가 1.7초가 됐다. 그 지연 때문에 1단계 임계값을
0.10 아래로 못 내렸고, 실기 재현율이 41% 에서 멈췄다.

`하이 티드`는 이 사슬의 첫 고리가 없다. 실측(12화자 x 3속도, whisper medium, 힌트 없이):

| 문구 | 자모거리 중앙 | 최대 | ≤0.25 통과 |
|---|---|---|---|
| **하이 티드** | **0.000** | 0.222 | **100%** |
| 재하봇(대조) | 0.429 | 0.714 | 11% |

36개 중 35개가 정확히 `하이티드`로 전사됐다. 재하봇은 **한 번도** 제대로 안 적혔다.

➡️ **그래서 이 모델(1단계)은 완벽할 필요가 없다.** 2단계 whisper 가 헛깨움을 거른다
(아이 말 12개 0건, 부모 말 8개 0건, 음운적 함정 20개 중 1건 = '하이브리드').
1단계는 **놓치지 않는 것**만 하면 된다. 임계값을 낮게 열 수 있다.

## v5 -> v6 데이터 설계 변경 세 가지

**① 화자를 실측 F0 대역으로 재편했다 (20종 -> 12종)**
v4 실패의 원인은 F0 분포였다(학습 중앙 165Hz vs 아버님 104~116Hz). v5 는 이걸
'피치를 낮추는 증강'으로 풀려다 다른 칸의 몫을 빼앗아 실패했다. **증강이 아니라
화자를 골라서** 푼다. 실측 F0: 저역 M5 97 / M2+M5 99 / M3 122, 고역 F2 327 / F1 308.

**② 합성 속도 상한 1.05 -> 1.15**
'하이 티드'는 1.15 에서도 QC 통과 80%(재하봇은 42~50%). 1.15 는 **진짜로 빨리 발음한
소리**라 WSOLA 로 늘린 가짜 빠르기와 다르다 — v4 실기에서 '빠르게'가 10% 였던 게 그 차이였다.
수율이 낮은 칸이라 문구별로 부족분을 계산해 보충 생성했다.

**③ 증강 격자 11칸 -> 10칸, 낮추는 칸 0.85 -> 0.80**
7칸(v4)에는 낮추는 칸이 없어 증강이 분포를 위로 민다(≤116Hz 19.0% -> 12.4%).
낮추는 칸의 역할은 '저역을 더하는 것'이 아니라 **'지키는 것'**이다.
그리고 `pitch_shift` 는 내리는 쪽에서만 명령보다 덜 내려간다(0.85 -> 실제 0.908).
의도한 0.85 를 얻으려고 **0.80 을 명령**한다.

## 🔴 시험 방식이 바뀌었다 — 여기가 가장 중요하다

**v1~v5 는 시험지와 교과서가 같은 곳에서 나왔다.** 같은 생성기·같은 화자·같은 문구를
무작위로 잘라 train/test 로 썼다. 그래서 오프라인 91% 가 실기 30% 였다. 석 달간
'호출어를 알아듣는가'가 아니라 **'합성 음성을 외웠는가'**를 재고 있었던 것이다.

v6 는 **화자를 통째로 빼서** 나눴다:

- `M3` 122Hz — 저역. 아버님(104~116Hz)에게 가장 가깝고 **v4 가 무너진 자리**다.
- `F2+F5` 258Hz — 고역. 재하 음역대(미측정, 추정).

학습에는 이 둘이 **한 클립도 없다.** 그래서 아래 재현율은 '외웠는가'가 아니라
**'처음 듣는 목소리에도 되는가'**를 잰다. 증강본도 원본과 같은 쪽에만 둔다(누수 검사 통과).

⚠️ 그래도 **합성음이다.** 최종 판정은 실기다. 이 노트북의 숫자로 '된다'고 말하지 말 것.


In [ ]:
!nvidia-smi
import sys; print('python', sys.version)

In [ ]:
# ── 경로·토큰 ──────────────────────────────────────────────
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # 배경음(MUSAN) 다운로드 rate limit 회피

MODEL_NAME = 'jaehabot_v6'          # 이전 산출물을 덮어쓰지 않는다
WORK   = '/content/jaeha_wake'
DATA   = os.path.join(WORK, 'data')
OUTDIR = os.path.join(WORK, 'output')
OUT    = os.path.join(OUTDIR, MODEL_NAME)
BACKUP = '/content/drive/MyDrive/jaeha_wake_backup'
DATA_TAR = os.path.join(BACKUP, 'jaeha_wake_data.tar')
OUR_TAR  = os.path.join(BACKUP, 'jaehabot_v6.tar.gz')       # PC 에서 만든 우리 데이터(341MB)

for d in (WORK, DATA, OUTDIR, BACKUP):
    os.makedirs(d, exist_ok=True)
os.makedirs(os.path.join(WORK, 'configs'), exist_ok=True)
os.chdir(WORK)

print('우리 데이터:', OUR_TAR, '->', '있음' if os.path.exists(OUR_TAR) else '🔴 없음! Drive 에 올렸는지 확인')
!df -h /content | cat

In [ ]:
# ── 설치 (이미 깔려 있으면 건너뜀) ──────────────────────────
import importlib.util

if importlib.util.find_spec('livekit') is None:
    !apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox
    !pip -q install 'livekit-wakeword[train,eval,export,voxcpm]'
else:
    print('livekit-wakeword 이미 설치됨 — 건너뜀')

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU!')

In [ ]:
# ── 진행이 보이는 실행 헬퍼 (셸매직은 출력이 버퍼링돼 죽었는지 도는지 알 수 없다) ──
import subprocess, sys, time
from IPython.display import clear_output

def run_stream(*args, every=15.0, keep_tail=8):
    p = subprocess.Popen(list(args), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail, n, last = [], 0, 0.0
    for line in p.stdout:
        tail.append(line.rstrip())
        if len(tail) > keep_tail: tail.pop(0)
        n += 1
        if time.time() - last >= every:
            clear_output(wait=True)
            print(f'[{n}줄 진행중] {" ".join(args[:2])}')
            print('\n'.join(tail)); sys.stdout.flush()
            last = time.time()
    p.wait()
    clear_output(wait=True)
    print('\n'.join(tail))
    print(f'\n--- exit code: {p.returncode} (총 {n}줄) ---')
    if p.returncode != 0:
        raise RuntimeError(f'{args[0]} 실패(exit {p.returncode}) — 위 로그 확인')

## 1. config

v2 설정을 그대로 쓰되 네 가지만 바꾼다.

- `model_name` → `jaehabot_v6` (이전 산출물 보존)
- `target_phrases` → **`하이 티드 / 하이티드 / 하이 티드야`** (호출어 교체)
- `custom_negative_phrases` → 하이 티드용 85개로 교체.
  옛 목록은 재하봇용(`자동차`·`로봇`·`보트`·`재현이`)이라 새 호출어와 아무 관계가 없다.
- **`n_samples` 50** — voxcpm 으로 음성을 만들 생각이 없다. generate 를 돌리는
  목적은 오직 **배경음(MUSAN) 폴더를 만드는 것**이고, 거기서 딸려 나오는
  voxcpm 긍정·부정은 다음 장에서 우리 데이터로 통째로 덮어쓴다.

⚠️ 아래 셀에서 `cfg[...] = ...` 로 덮어쓰는 값이 **YAML 보다 우선한다.**
   v6 로 옮길 때 YAML 만 고치고 이 줄을 놓쳐서 하마터면 '재하봇'으로 학습할 뻔했다.


In [ ]:
import yaml

BASE_YAML = r'''
model_name: jaehabot_v6
# ============================================================================
# v6 (2026-08-25) — 호출어가 '재하봇' -> '하이 티드' 로 바뀌었다.
#
# ⚠️ 아래 target_phrases / custom_negative_phrases 는 **이 노트북에서 쓰이지 않는다.**
#    generate 단계는 배경음 폴더를 만들려고만 돌리고, 긍정·부정은 3장에서 우리 tar 로
#    통째로 갈아 끼운다. 그래도 적어 두는 이유는 (a) 설정과 데이터가 어긋나 보이면
#    다음 사람이 혼란스럽고 (b) 언젠가 generate 산출물을 쓸 때 기준이 필요해서다.
#
# 🔴 tts_backend 는 voxcpm 이지만 **우리 긍정·부정은 Supertonic 으로 만들었다.**
#    voxcpm 은 2026-08-07 에 기각됐다(짧은 문구 통과율 40%, 폭주 10~45%).
# ============================================================================
target_phrases:
- 하이 티드
- 하이티드        # 붙여 쓴 형태 — 어절 사이 쉼이 없어 빨리 부르는 모양이 된다
- 하이 티드야     # 호격
tts_backend: voxcpm
# 프롬프트가 6 -> 16 으로 늘어 프롬프트당 샘플 수가 1/3 수준으로 얇아진다.
# 2000 을 유지하면 프롬프트당 125개뿐 → 3000 으로 올려 187개를 확보한다.
# ⚠️ 생성 시간이 비례해 늘어난다(voxcpm 은 순차 생성, 2000개 ≈ 2.5~3시간 → 3000개 ≈ 4~4.5시간).
#    시간이 부족하면 2000 으로 되돌리되, 프롬프트 수를 12개로 줄이는 쪽을 먼저 검토할 것.
n_samples: 3000
n_samples_val: 750
n_background_samples: 3000
n_background_samples_val: 750
tts_batch_size: 256     # ⚠️ voxcpm_backend.py 가 `del batch_size` 로 무시한다(순차 생성). 무해해서 남겨둠.
custom_negative_phrases:
- 하이
- 하이요
- 하이 하이
- 안녕 하이
- 네 하이
- 하이고
- 하이브리드 차
- 하이브리드
- 하이킹 갈까
- 하이킹
- 하이라이트 봤어
- 하이라이트
- 하이파이브 하자
- 하이파이브
- 하이텐션
- 하이든 음악
- 하이힐
- 하이애나
- 하이볼
- 하이패스
- 티드
- 티드가
- 티비
- 티비 보자
- 티셔츠
- 티나
- 그 다음에 티
- 티도 안 나
- 타이드 세제
- 타이드
- 다이어트 해야지
- 다이어트
- 하이드
- 차이 나는
- 사이다
- 하이라이터
- 하이 준비
- 티드 아니야
- 하이 그만
- 안 티드
- 하이 어디야
- 엄마 어디 있어
- 아빠 안아줘
- 이거 뭐야
- 이거 줘
- 물 줘
- 더 줘
- 안 해
- 싫어 싫어
- 무서워
- 같이 놀자
- 멍멍이다
- 빵빵 자동차
- 쉬 마려워
- 배고파
- 졸려
- 아파
- 없어
- 어디 갔어
- 나도 나도
- 내 거야
- 하지 마
- 그만
- 좋아
- 재밌다
- 또 해줘
- 안녕히 계세요
- 잘 자
- 맘마 줘
- 밥 먹자 이리 와
- 손 씻고 오세요
- 신발 신자
- 잠깐만 기다려
- 이거 정리하고 자자
- 티비 그만 보고
- 오늘 어린이집 재밌었어
- 물 마실래
- 옷 갈아입자
- 빨리 와야지
- 안 돼 위험해
- 착하다 우리 아기
- 누구세요
- 잠깐만요
- 알겠습니다
- 감사합니다
noise_scales:
- 0.98
noise_scale_ws:
- 0.98
length_scales:          # ⚠️ Piper 전용 — voxcpm 에선 무시됨. 속도는 프롬프트로만 제어된다.
- 0.75
- 1.0
- 1.25
slerp_weights:
- 0.2
- 0.35
- 0.5
- 0.65
- 0.8
piper_tts:
  checkpoint_relpath: piper/en-us-libritts-high.pt
voxcpm_tts:
  model_id: openbmb/VoxCPM2
  model_cache_relpath: voxcpm/VoxCPM2
  local_model_path: null
  load_denoiser: false
  cfg_values:
  - 1.5
  - 2.0
  - 2.5
  - 3.0
  inference_timesteps_list:   # 기본 [8,10,12] 에서 낮춤 — 생성 속도의 유일한 실질 레버
  - 4
  - 6
  - 8
  # ⚠️ 반드시 voxcpm_tts 아래에 중첩할 것. 최상위에 쓰면 조용히 무시되고 원본 영어 33종이 쓰인다.
  voice_design_prompts:
  # --- 빠름 (7개) — v1 에 없던 구간. 실기 실패의 원인이라 가장 두껍게 ---
  - 빠르게 부르는 한국인 어린 여자아이
  - 급하게 외치듯 부르는 한국인 남자아이
  - 신이 나서 빠르게 말하는 어린아이
  - 다급하게 부르는 어린아이
  - 빠른 말투로 또렷하게 말하는 한국인 성인 여성
  - 빠르게 툭 내뱉듯 부르는 어린아이
  - 숨차게 빠르게 부르는 한국인 아이
  # --- 보통 (6개) — v1 프롬프트를 속도 표현만 붙여 승계 ---
  - 보통 속도로 밝고 높게 말하는 한국인 어린 여자아이
  - 보통 속도로 해맑게 말하는 한국인 남자아이
  - 보통 속도로 또렷하게 말하는 한국인 성인 여성
  - 보통 속도로 차분하게 말하는 한국인 성인 남성
  - 보통 속도로 조금 웅얼거리며 말하는 어린아이
  - 신나서 크게 부르는 한국인 어린아이
  # --- 느림 (3개) — v1 이 사실상 이 구간이라 이미 잘 된다. 회귀 방지 최소분만 유지 ---
  - 아주 천천히 또박또박 부르는 한국인 어린아이
  - 천천히 조심스럽게 부르는 한국인 여자아이
  - 느릿하게 웅얼거리는 어린아이
data_dir: /content/jaeha_wake/data
output_dir: /content/jaeha_wake/output
augmentation:
  clip_duration: 2.0
  batch_size: 64
  rounds: 3
  background_paths:
  - ./data/backgrounds
  rir_paths:
  - ./data/rirs
model:
  model_type: conv_attention
  model_size: medium
steps: 50000
learning_rate: 0.0001
weight_decay: 0.01
label_smoothing: 0.05
max_negative_weight: 3000
target_fp_per_hour: 0.1
batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 1024
  background_noise: 50
'''

cfg = yaml.safe_load(BASE_YAML)
cfg['model_name'] = MODEL_NAME
cfg['data_dir'], cfg['output_dir'] = DATA, OUTDIR

# 🔴 이 줄이 YAML 을 덮어쓴다. 호출어를 바꿀 때 **여기도** 고쳐야 한다.
#    (v6 작업 중 YAML 만 고치고 이 줄을 놓쳐 '재하봇'으로 학습할 뻔했다)
cfg['target_phrases'] = ['하이 티드', '하이티드', '하이 티드야']

# voxcpm 합성은 최소로 — 배경음 폴더만 얻으면 된다
cfg['n_samples'], cfg['n_samples_val'] = 50, 10
cfg['n_background_samples'], cfg['n_background_samples_val'] = 3000, 750

CFG_PATH = 'configs/jaeha_v6.yaml'
yaml.safe_dump(cfg, open(CFG_PATH, 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('저장:', CFG_PATH)
print('  model_name  :', cfg['model_name'])
print('  호출어      :', cfg['target_phrases'])
print('  부정 문구   :', len(cfg['custom_negative_phrases']), '개')
print('  n_samples   :', cfg['n_samples'], '(배경음용 최소값 — 우리 데이터로 덮어쓴다)')
print('  augment     :', cfg['augmentation']['rounds'], '라운드')
print('  steps       :', cfg['steps'])

In [ ]:
# ── 데이터 준비 (있으면 건너뛴다) ───────────────────────────
have_feat = os.path.isdir(os.path.join(DATA, 'features'))
have_bg   = os.path.isdir(os.path.join(DATA, 'backgrounds'))

if have_feat and have_bg:
    print('데이터 이미 있음 — 다운로드 건너뜀')
elif os.path.exists(DATA_TAR):
    print('Drive 캐시에서 복원:', DATA_TAR)
    run_stream('tar', '-C', WORK, '-xf', DATA_TAR)
else:
    print('데이터 없음 → setup 실행 (ACAV100M 특징 16GB 등, 수십 분)')
    run_stream('livekit-wakeword', 'setup', '--config', CFG_PATH)

!du -sh {DATA}/* 2>/dev/null | cat

## 2. generate — **배경음 폴더를 만들려고** 돌린다 (10분 안팎)

여기서 나오는 `positive_*` · `negative_*` 는 voxcpm 산이라 쓸 물건이 아니다.
다음 셀에서 통째로 지우고 우리 것으로 바꾼다.

In [ ]:
run_stream('livekit-wakeword', 'generate', CFG_PATH, every=20, keep_tail=5)

import glob
for d in sorted(glob.glob(OUT + '/*')):
    if os.path.isdir(d):
        print(f'{len(glob.glob(d + "/*.wav")):7d}  {os.path.basename(d)}')

## 3. ★ 우리 데이터로 교체 — 여기가 이 노트북의 핵심

voxcpm 이 만든 긍정·부정을 **지우고** 젯슨에서 검수까지 끝낸 것을 푼다.
배경음(`background_*`)은 실제 소음 데이터셋이라 그대로 둔다.

In [ ]:
import shutil, glob

# 1) generate 가 만든 긍정·부정 제거 (남겨두면 검수 안 된 음성이 학습에 섞인다)
for sub in ['positive_train', 'positive_test', 'negative_train', 'negative_test']:
    p = os.path.join(OUT, sub)
    n = len(glob.glob(p + '/*.wav'))
    shutil.rmtree(p, ignore_errors=True)
    print(f'제거 {sub}: {n}개')

# 2) 우리 데이터 풀기 (tar 안이 jaehabot_v6/... 구조라 OUTDIR 에 푼다)
assert os.path.exists(OUR_TAR), f'없음: {OUR_TAR} — Drive 에 올렸는지 확인'
print('\n복원 중(수 분):', OUR_TAR)
run_stream('tar', '-C', OUTDIR, '-xzf', OUR_TAR)

print()
# 🔴 v6 개수. 화자 홀드아웃(M3 · F2+F5)으로 나눠서 v5 와 비율이 다르다(시험 약 17%).
EXPECT = {'positive_train': 6608, 'positive_test': 1340,
          'negative_train': 8348, 'negative_test': 1648}
ok = True
for sub, want in EXPECT.items():
    got = len(glob.glob(os.path.join(OUT, sub, 'clip_*.wav')))
    mark = 'OK ' if got == want else '🔴'
    ok = ok and got == want
    print(f'{mark} {sub:<16} {got:6d} (기대 {want})')
for d in sorted(glob.glob(OUT + '/background*')):
    print(f'    {os.path.basename(d):<16} {len(glob.glob(d + "/*.wav")):6d}')
assert ok, '개수가 다르다 — tar 가 깨졌거나 덜 풀렸다'

# 시험 화자가 학습에 안 섞였는지 확인 — 이 검사가 v6 의 핵심이다.
import json
def voices(sub):
    p = os.path.join(OUT, sub, 'manifest.jsonl')
    return {json.loads(x)['voice'] for x in open(p, encoding='utf-8') if x.strip()}
tr, te = voices('positive_train'), voices('positive_test')
print(f'\n학습 화자 {len(tr)}종 / 시험 화자 {sorted(te)}')
assert not (tr & te), f'🔴 시험 화자가 학습에 섞였다: {tr & te} — 자기채점이 된다'
print('✅ 화자 홀드아웃 확인 — 시험 화자는 학습에 한 클립도 없다')
print('\n교체 완료')


### 3-2. 들어보고 넘어간다

v2 의 실패는 **아무도 안 들어본 것**이었다. 4시간짜리 학습을 걸기 전에 확인한다.

In [ ]:
import random, soundfile as sf
from IPython.display import Audio, display

for sub in ['positive_train', 'negative_train']:
    clips = sorted(glob.glob(os.path.join(OUT, sub, 'clip_*.wav')))
    print(f'=== {sub} ({len(clips)}개) 중 3개 ===')
    for c in random.Random(0).sample(clips, 3):
        y, sr = sf.read(c)
        print(' ', os.path.basename(c), f'{len(y)/sr:.2f}s')
        display(Audio(y, rate=sr))

## 4. augment — 소음·잔향·EQ (20~40분)

내장 증강은 EQ·왜곡·RIR잔향·배경믹싱이다. 피치/속도는 없지만,
우리는 그 두 축을 젯슨에서 이미 넣었다(긍정·부정 대칭).

In [ ]:
run_stream('livekit-wakeword', 'augment', CFG_PATH, every=30)

In [ ]:
# augment 검증 — _rN 이 0이면 학습 데이터가 비어 있는 것이므로 train 금지
for sub in ['positive_train', 'negative_train']:
    d = os.path.join(OUT, sub)
    tot = len(glob.glob(d + '/*.wav'))
    aug = len(glob.glob(d + '/*_r*.wav'))
    print(f'{sub:<16} 전체 {tot:6d} / 증강본 {aug:6d}')
    assert aug > 0, f'{sub} 증강본 없음 — 이 상태로 train 금지'
print('OK')

### 4-2. 💾 백업 (권장)

여기까지가 젯슨 50분 + Colab augment 결과다. 런타임이 끊기면 다시 만들어야 한다.
(우리 원본 tar 은 Drive 에 있으니 최악이어도 3번부터 다시 하면 된다.)

In [ ]:
DO_BACKUP = True
TAR = os.path.join(BACKUP, f'{MODEL_NAME}_augmented.tar')
if DO_BACKUP:
    run_stream('tar', '-C', OUTDIR, '-cf', TAR, MODEL_NAME)
    !ls -lh {TAR} | cat
else:
    print('건너뜀')

## 5. train (수 시간)

In [ ]:
run_stream('livekit-wakeword', 'train', CFG_PATH, every=60)

## 6. export — ONNX

⚠️ `--quantize`(int8)는 shape inference 에러로 실패한다. 젯슨에 int8 은 불필요.

In [ ]:
run_stream('livekit-wakeword', 'export', CFG_PATH)

## 7. 공식 평가

실제 음성 ~18시간(ACAV100M 검증셋)을 부정으로 써서 지표를 낸다.
**v1 기준선: AUT 0.0302 / FPPH 0.22 / Recall 40.1% @ thr 0.50**

In [ ]:
run_stream('livekit-wakeword', 'eval', CFG_PATH)

## 8. ★ threshold 스윕 — 젯슨에 넣을 값을 고른다

🔴 **v1~v5 의 recall 수치와 비교하지 말 것.** 호출어가 다르고(재하봇 vs 하이 티드),
무엇보다 **시험 방식이 다르다.** v1~v5 는 학습과 같은 화자로 시험했고(자기채점),
v6 는 학습에 한 번도 안 쓴 화자(`M3` 122Hz · `F2+F5` 258Hz)로만 시험한다.
같은 자로 잰 값이 아니므로 숫자를 나란히 놓으면 틀린 결론이 난다.

**v6 에서 값을 고르는 기준이 달라졌다.** 이제 2단계 whisper 검증이 헛깨움을 거른다
(합성 실측: 아이 말 12개 0건, 부모 말 8개 0건, 음운적 함정 20개 중 1건).
그래서 1단계는 **놓치지 않는 쪽**으로 낮게 연다.

| 목표 | 값 |
|---|---|
| recall | 높을수록 좋다 — 놓친 호출은 2단계가 되살릴 수 없다 |
| FPPH | **관대하게 봐도 된다.** 후보가 늘어도 2단계가 거른다 |
| 상한 | 검증 1회가 약 0.85초다. FPPH 가 너무 높으면 실시간이 포화된다 — 그게 진짜 상한 |

➡️ FPPH 가 **시간당 20~40** 수준까지는 감당된다(재하봇 실기에서 유튜브 켠 채 20/시간이었다).
   그 범위에서 recall 이 가장 높은 값을 고른다.


In [ ]:
import numpy as np, onnxruntime as ort

sess = ort.InferenceSession(OUT + f'/{MODEL_NAME}.onnx', providers=['CPUExecutionProvider'])
iname = sess.get_inputs()[0].name

def load_feats(path):
    a = np.load(path, mmap_mode='r')
    if a.ndim == 2:
        n = (len(a) // 16) * 16
        a = np.asarray(a[:n]).reshape(-1, 16, 96)
    return a

def scores(path, bs=4096):
    a = load_feats(path); out = []
    for i in range(0, len(a), bs):
        x = np.asarray(a[i:i+bs], dtype=np.float32)
        out.append(sess.run(None, {iname: x})[0].reshape(-1))
    return np.concatenate(out)

ps = scores(OUT + '/positive_features_test.npy')
ns = np.concatenate([scores(f) for f in [
    OUT + '/negative_features_test.npy',
    OUT + '/background_noise_features_test.npy',
    DATA + '/features/validation_set_features.npy']])
hours = len(ns) * 2.0 / 3600
print(f'긍정 {len(ps)} / 부정 {len(ns)} ({hours:.2f}시간)\n')

# 🔴 v1 비교표를 뺐다. 호출어도 다르고 시험 방식도 다르다(v6 는 화자 홀드아웃).
#    나란히 놓으면 틀린 결론이 난다.
print(f'{"thr":>6} {"recall":>8} {"FPPH":>9}   비고')
for t in [0.5,0.4,0.3,0.25,0.2,0.15,0.12,0.1,0.08,0.06,0.05,0.04,0.03,0.02]:
    r, f = (ps>=t).mean()*100, (ns>=t).sum()/hours
    note = ''
    if f > 40:
        note = '검증 포화 위험(1회 0.85초)'
    elif 20 <= f <= 40:
        note = '<- 이 구간에서 recall 최대인 값'
    print(f'{t:6.2f} {r:7.1f}% {f:9.2f}   {note}')
print('\n2단계가 헛깨움을 거르므로 FPPH 는 관대하게 본다. 상한은 실시간 포화다.')

## 9. 젯슨 배포용 zip (추론엔 ONNX 3개가 필요하다)

In [ ]:
import importlib.util
EXP = '/content/jaeha_wake_export_v5'
shutil.rmtree(EXP, ignore_errors=True); os.makedirs(EXP)

for f in glob.glob(OUT + '/**/*.onnx', recursive=True): shutil.copy(f, EXP)
for f in glob.glob(OUT + '/**/*.pt', recursive=True):   shutil.copy(f, EXP)

spec = importlib.util.find_spec('livekit.wakeword')
pkg = os.path.dirname(spec.origin) if spec and spec.origin else ''
for name in ['melspectrogram.onnx', 'embedding_model.onnx']:
    hits = glob.glob(pkg + '/**/' + name, recursive=True)
    print(name, '->', hits[:1])
    if hits: shutil.copy(hits[0], EXP)
    else:    print(f'  [!] {name} 못 찾음 — 젯슨 추론에 필요하니 수동 확인')

shutil.copy(CFG_PATH, EXP)
zip_local = shutil.make_archive('/content/jaeha_wake_export_v5', 'zip', EXP)
shutil.copy(zip_local, BACKUP)
print('\n묶음:', sorted(os.listdir(EXP)))
print('Drive 사본:', os.path.join(BACKUP, os.path.basename(zip_local)))

## 10. 다음 (젯슨)

1. zip 을 `~/jaeha_bot/models/wake/v6/` 에 푼다 — **v4 를 덮어쓰지 말 것**(롤백용).
2. `configs/model_paths.yaml` 의 wake 블록:
   ```yaml
   wake:
     word: 하이티드          # 이미 바뀌어 있다(공백 없는 형태를 쓴다)
     onnx:
       model_dir: models/wake/v6
       classifier: jaehabot_v6.onnx
       threshold: <8번 스윕에서 고른 값>
       trigger_frames: 1
   ```
   ⚠️ 지금 이 파일에는 **"분류기가 아직 재하봇용 v4 라 안 깨어난다"**는 경고 주석이
   달려 있다. v6 를 넣으면서 그 주석도 지울 것.
3. **실기 확인이 진짜 판정이다.** 또박또박 / 보통 / 빠르게 / 흘려서 각 10회.
   지금까지의 recall 은 전부 합성음 점수다.
4. 2단계 검증 값도 같이 본다 (`configs/model_paths.yaml` 의 `wake.onnx.verify`):
   - `plain_max_ratio: 0.35` — 합성 실측으로 정한 값이다. **실음성으로 다시 잴 것.**
     호출어는 ≤0.222, 가장 가까운 함정('하이브리드')은 0.300 이었다.
   - 힌트 없이도 읽히므로 **2패스를 1패스로 줄일 수 있다**(1.7초 -> 0.85초).
     아직 안 했다. 줄이면 임계값을 더 낮게 열 수 있다.

> 🔴 **이 모델이 잘 나와도 다음 한 걸음은 실음성 녹음이다.**
> 학습 데이터에 실제 사람 목소리가 **한 건도 없다**. Apple 의 Hey Siri 는 부엌·차·침실·
> 식당에서 녹음한 50만 발화로 학습했다. 우리는 합성 2,500개다.
> 다만 우리는 시리가 될 필요가 없다 — **한 가정 3~4명**에게만 되면 된다.
> 그래서 수백 개면 된다: 재하 30~50 / 부모 20~30 / 거실 소음 1시간.
> 임베딩은 고정이고 분류기만 바꾸므로 적은 데이터로도 효과가 크다.


## 9-4. 🔴 격자 칸별 재현율 — 평균은 실패를 숨긴다

위 스윕은 **전체 평균**이라 특정 조건만 못 잡아도 숫자가 좋게 나온다.
v3 가 정확히 그렇게 실패했고(오프라인 91% / 실기에서 빠른 발화 놓침),
**v4 는 이 검사를 20%p 기준으로 '통과'했는데도 실기에서 실패했다** — 최악 칸이
19%p 라 1%p 차이로 빠져나갔고, 문제는 한 칸이 튄 게 아니라 **한 축이 통째로 기운
것**이었다(시간 1.35 인 칸 둘 다 65~67%). 그래서 칸과 **축별 평균**을 같이 본다.

**v6 에서 이 검사가 전보다 의미 있는 이유**: 시험 클립이 학습에 없는 화자
(`M3` 122Hz · `F2+F5` 258Hz)에서만 나온다. 즉 '외운 걸 다시 묻는' 게 아니라
**처음 듣는 목소리에서도 그 조건이 되는가**를 잰다.

이번에 꼭 볼 칸:
- **피치 0.8** = 성인 남성 음역 ← 여기가 낮으면 아버님 목소리를 또 놓친다
- **시간 1.35** = 빠른 발화 ← v4 가 약했던 축(65~67%)
- **시간 1.0 / 피치 1.35** = 아이 목소리 ← **떨어지면 안 된다**

⚠️ 그래도 **합성음이다.** 화자를 뺐어도 Supertonic 이 만든 소리라는 건 같다.
   최종 판정은 실기 — `tools/record_wake_real.py`.


In [ ]:
# 내보낸 ONNX 3종을 런타임과 같은 순서로 태워 원본 wav 를 채점한다.
# (app/wake_onnx.py 와 같은 계산 — 멜은 int16 스케일 입력 + x/10+2 정규화)
import json, glob, numpy as np, soundfile as sf, onnxruntime as ort
from collections import defaultdict

P = ['CPUExecutionProvider']
mel_s = ort.InferenceSession(EXP + '/melspectrogram.onnx', providers=P)
emb_s = ort.InferenceSession(EXP + '/embedding_model.onnx', providers=P)
cls_s = ort.InferenceSession(EXP + f'/{MODEL_NAME}.onnx', providers=P)
MEL_WINDOW, EMB_WINDOW, MEL_BANDS, EMB_DIM, FRAME = 76, 16, 32, 96, 1280

def best_score(y):
    pad = (np.random.randn(32000) * 1e-4).astype(np.float32)
    s = np.concatenate([pad, y.astype(np.float32), pad[:8000]])
    mel_buf, emb_buf, best = [], [], 0.0
    for i in range(0, len(s) - FRAME, FRAME):
        a = (s[i:i+FRAME].reshape(1, -1) * 32767).astype(np.float32)
        m = np.squeeze(mel_s.run(None, {mel_s.get_inputs()[0].name: a})[0])
        m = m.reshape(-1, MEL_BANDS) / 10.0 + 2.0
        mel_buf.extend(m.astype(np.float32))
        mel_buf = mel_buf[-MEL_WINDOW:]
        if len(mel_buf) < MEL_WINDOW:
            continue
        w = np.stack(mel_buf)[None, :, :, None].astype(np.float32)
        e = np.squeeze(emb_s.run(None, {emb_s.get_inputs()[0].name: w})[0])
        emb_buf.append(e.reshape(EMB_DIM).astype(np.float32))
        emb_buf = emb_buf[-EMB_WINDOW:]
        if len(emb_buf) < EMB_WINDOW:
            continue
        f = np.stack(emb_buf)[None, :, :].astype(np.float32)
        best = max(best, float(np.squeeze(
            cls_s.run(None, {cls_s.get_inputs()[0].name: f})[0])))
    return best

man = {}
with open(OUT + '/positive_test/manifest.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            r = json.loads(line); man[r['clip']] = r

cells = defaultdict(list)
for nm, r in man.items():
    cells[(r.get('rate', 1.0), r.get('pitch', 1.0))].append(nm)

rng = np.random.RandomState(0)
THRS = [0.30, 0.25, 0.20, 0.15]
print(f"{'시간':>5} {'피치':>5} {'n':>4} " + ' '.join(f'{t:>7.2f}' for t in THRS))
print('-' * 46)
allsc = []
for key in sorted(cells):
    names = cells[key]
    pick = [names[i] for i in rng.choice(len(names), min(60, len(names)), replace=False)]
    sc = []
    for nm in pick:
        y, _ = sf.read(OUT + '/positive_test/' + nm, dtype='float32')
        sc.append(best_score(y.mean(axis=1) if y.ndim > 1 else y))
    sc = np.array(sc); allsc.append((key, sc))
    print(f'{key[0]:>5} {key[1]:>5} {len(sc):>4} ' +
          ' '.join(f'{(sc>=t).mean()*100:>6.0f}%' for t in THRS))

base = np.concatenate([s for _, s in allsc])
print(f"\n{'전체':>11} {len(base):>4} " +
      ' '.join(f'{(base>=t).mean()*100:>6.0f}%' for t in THRS))

# ── 판정 ──────────────────────────────────────────────────────────────
# v4 의 교훈: '최악 칸 하나'만 보면 축이 통째로 기운 걸 놓친다. 칸·축 둘 다 본다.
B = (base >= 0.25).mean()
rows = sorted(((k, (s >= 0.25).mean()) for k, s in allsc), key=lambda kv: kv[1])
print('\n낮은 칸 3개 (thr 0.25)')
for k, v in rows[:3]:
    print(f'   시간 {k[0]:<5} 피치 {k[1]:<5} {v*100:>3.0f}%  (전체보다 {(B-v)*100:>2.0f}%p 낮음)')

print('\n축별 평균 (thr 0.25)')
bad_axis = []
for name, idx in (('시간', 0), ('피치', 1)):
    vals = {}
    for k, s in allsc:
        vals.setdefault(k[idx], []).append((s >= 0.25).mean())
    for v in sorted(vals):
        m = float(np.mean(vals[v]))
        flag = '  🔴' if B - m > 0.15 else ''
        if flag:
            bad_axis.append(f'{name} {v}')
        print(f'   {name} {v:<5} {m*100:>3.0f}%{flag}')

worst_k, worst_v = rows[0]
gap = B - worst_v
print(f'\n최악 칸: 시간 {worst_k[0]} 피치 {worst_k[1]} -> {worst_v*100:.0f}% '
      f'(전체 {B*100:.0f}% 보다 {gap*100:.0f}%p 낮음)')
if gap > 0.20 or bad_axis:
    print('🔴 배포하지 말 것. ' +
          (f'기운 축: {", ".join(bad_axis)}. ' if bad_axis else '') +
          '학습 데이터를 고쳐야 한다.')
else:
    print('✅ 칸·축 모두 큰 쏠림이 없다.')
print('\n⚠️ 이 수치는 합성음이다. 화자는 홀드아웃이라 자기채점은 아니지만,'
      ' 여전히 Supertonic 이 만든 소리다.')
print('   최종 판정은 실기 — tools/record_wake_real.py 로 조건당 10회 이상.')